<a href="https://colab.research.google.com/github/anamta-ansari/flyrank-ai/blob/main/Work/Notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This notebook turns the Week-5/Week-6 modeling and validation work into a practical, human-reviewed
content action playbook. Simple words, honest numbers.

**Skill loaded for this task:** `writing-honest-claims` (claim ladder: observed → directional →
decision-support, never causal without a design) and `flyrank/flyrank-data` (starter dataset gotchas).


## 1. Ranked actions + reason codes
The queue: what to do first, and why, in words a human trusts.

**Why this queue is not "the model's score."** Two things earlier in this project block the raw
ML output from being handed to an editor as a priority number:

- **Week-5 label leakage.** `refresh_label` was built directly from `content_age_days`, `ctr`, and
  `impressions_90d` — and all three columns were then kept in the feature set. The Random Forest
  didn't learn a generalizable pattern; it re-derived the rule that made the label. That's the real
  reason accuracy landed at 0.9995 with only 3/6,000 misclassified — not evidence the model is
  production-ready.
- **Week-6 honest-split collapse — with a root cause.** The grouped/time-aware split reported
  Accuracy 1.0 but Precision/Recall/F1 = 0. Tracing it down: `is_declining` was built with
  `trend_direction.str.contains("declin")`, but the real values in this column are
  `{down, stable, up, new, flat}` — none contain the substring "declin". That filter matches **0 of
  30,000 rows**, so the classifier was trained and evaluated against an all-zero target. The
  degenerate metrics are a labeling bug, not a finding about the model's honest-split performance.
  (Corrected below to `trend_direction == "down"`.)

Given that, this playbook does **not** rank content by a model probability. It ranks by a small set
of **transparent, auditable reason codes** computed straight from the trailing-90-day metrics — each
flagged item shows *exactly* why it's on the list, which is what a person reviewing the queue
actually needs.


In [9]:
# Setup — same pattern as Weeks 4-6: clone the starter repo (or use local data/ if already present)
import os, sys, subprocess, json
from datetime import datetime, timezone
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)


Loaded: (30000, 44)


In [10]:
# --- Reason-code thresholds: derived from the data itself, not hard-coded guesses ---

# avg_position == 0 means "no ranking data", per the data dictionary — never treat it as rank zero.
has_data_position = df["avg_position"] > 0

median_ctr            = df.loc[has_data_position, "ctr"].median()
median_search_volume  = df["search_volume"].median()
engagement_p25         = df.loc[df["days_with_sessions"] > 0, "engagement_rate"].quantile(0.25)

reason_frame = pd.DataFrame({
    "STALE":                      (df["content_age_days"] > 180) & (df["days_since_last_update"] > 180),
    # corrected declining-trend signal (see Week-6 label bug above): use the real category, not a substring match
    "DECLINING_TREND":            (df["trend_direction"] == "down"),
    "LOW_CTR_GOOD_POSITION":      has_data_position & (df["avg_position"] <= 20) & (df["ctr"] < median_ctr),
    "HIGH_VOLUME_LOW_VISIBILITY": (df["search_volume"] > median_search_volume) & (~has_data_position | (df["avg_position"] > 20)),
    "LOW_ENGAGEMENT":             (df["days_with_sessions"] > 0) & (df["engagement_rate"] < engagement_p25),
})

df["reason_codes"]   = reason_frame.apply(lambda r: [c for c in reason_frame.columns if r[c]], axis=1)
df["priority_score"] = reason_frame.sum(axis=1)

print("Reason-code fire rate (share of the 30,000 items):")
print(reason_frame.mean().round(3))
print()
print("Priority-score distribution (how many reason codes fired, per item):")
print(df["priority_score"].value_counts().sort_index())


Reason-code fire rate (share of the 30,000 items):
STALE                         0.006
DECLINING_TREND               0.542
LOW_CTR_GOOD_POSITION         0.286
HIGH_VOLUME_LOW_VISIBILITY    0.105
LOW_ENGAGEMENT                0.000
dtype: float64

Priority-score distribution (how many reason codes fired, per item):
priority_score
0     8394
1    15088
2     6470
3       48
Name: count, dtype: int64


In [11]:
# --- Archetype -> action mapping ---
# Archetype here = content_type (the only content-shape axis in this dataset) crossed with which
# reason code dominates. Kept deliberately simple and named plainly.

def map_action(row):
    codes, ctype = row["reason_codes"], row["content_type"]
    if not codes:
        return "No action -- monitor only"
    if ctype == "comparison article":
        # comparison content ages fast (prices/specs) -- always route to a person, regardless of score
        return "Editorial review required before any change (comparison content ages fast)"
    if "STALE" in codes or "DECLINING_TREND" in codes:
        return "Refresh & re-optimize (update facts, strengthen keyword/intent match)"
    if "LOW_CTR_GOOD_POSITION" in codes:
        return "Rewrite title tag & meta description; test the SERP snippet"
    if "HIGH_VOLUME_LOW_VISIBILITY" in codes:
        return "Technical/on-page SEO audit (indexing, internal links, schema)"
    if "LOW_ENGAGEMENT" in codes:
        return "Improve intro/structure and on-page content depth"
    return "Review -- mixed signal"

df["recommended_action"] = df.apply(map_action, axis=1)

# --- light cost/value framing (transparent proxies, not a dollar model) ---
action_cost_tier = {
    "No action -- monitor only": "none",
    "Rewrite title tag & meta description; test the SERP snippet": "low",
    "Review -- mixed signal": "low",
    "Improve intro/structure and on-page content depth": "medium",
    "Refresh & re-optimize (update facts, strengthen keyword/intent match)": "medium",
    "Editorial review required before any change (comparison content ages fast)": "medium",
    "Technical/on-page SEO audit (indexing, internal links, schema)": "high",
}
df["est_cost_tier"] = df["recommended_action"].map(action_cost_tier)
# value_proxy: bigger for content with more search demand and higher commercial value (cpc) -- a
# proxy for "worth reviewing first", not a revenue forecast
df["value_proxy"] = df["search_volume"].fillna(0) * (1 + df["cpc"].fillna(0))

archetype_table = (
    df.groupby(["content_type", "recommended_action"])
      .size().reset_index(name="n_items")
      .sort_values(["content_type", "n_items"], ascending=[True, False])
)
archetype_table


,content_type,recommended_action,n_items
0,comparison article,Editorial review required before any change (c...,626
1,comparison article,No action -- monitor only,71
2,feedly article,No action -- monitor only,1072
3,feedly article,"Refresh & re-optimize (update facts, strengthe...",601
4,feedly article,Rewrite title tag & meta description; test the...,423
6,keyword article,"Refresh & re-optimize (update facts, strengthe...",15354
5,keyword article,No action -- monitor only,7251
7,keyword article,Rewrite title tag & meta description; test the...,2740
8,keyword article,"Technical/on-page SEO audit (indexing, interna...",1862


In [12]:
# --- Build the ranked queue ---
queue_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "priority_score", "reason_codes", "recommended_action", "est_cost_tier", "value_proxy",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "trend_direction", "search_volume",
]
queue = (
    df[queue_cols]
    .sort_values(["priority_score", "value_proxy"], ascending=[False, False])
    .reset_index(drop=True)
)
queue.insert(0, "rank", queue.index + 1)

queue.head(15)


,rank,content_id,client_id,content_type,main_intent,priority_score,reason_codes,recommended_action,est_cost_tier,value_proxy,content_age_days,days_since_last_update,ctr,avg_position,trend_direction,search_volume
0,1,content_bbca724138f2,client_6208ef0f77,keyword article,transactional,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,7616.0,236,236,0.0,12.1,down,1600.0
1,2,content_24abafed9707,client_8722616204,keyword article,transactional,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,480.0,231,231,0.0,1.3,down,480.0
2,3,content_02b0d6e30129,client_19581e27de,keyword article,transactional,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,174.9,313,313,0.0,6.9,down,110.0
3,4,content_7a888d3d99c8,client_19581e27de,keyword article,transactional,3,"[STALE, DECLINING_TREND, HIGH_VOLUME_LOW_VISIB...","Refresh & re-optimize (update facts, strengthe...",medium,154.8,313,313,0.0,67.6,down,90.0
4,5,content_f4b3081037b3,client_8722616204,keyword article,transactional,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,90.0,237,231,0.0,5.0,down,90.0
5,6,content_4f241bad48a3,client_6208ef0f77,keyword article,commercial,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,35.0,236,236,0.0,19.1,down,10.0
6,7,content_94991fe6268c,client_19581e27de,keyword article,commercial,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,11.2,313,313,0.0,12.4,down,10.0
7,8,content_149c671a2a91,client_6208ef0f77,keyword article,transactional,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,10.0,236,235,0.0,19.3,down,10.0
8,9,content_a34d943a132c,client_d029fa3a95,keyword article,informational,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,0.0,232,183,0.0,12.7,down,0.0
9,10,content_2e2a634851a0,client_d029fa3a95,keyword article,informational,3,"[STALE, DECLINING_TREND, LOW_CTR_GOOD_POSITION]","Refresh & re-optimize (update facts, strengthe...",medium,0.0,232,183,0.0,10.6,down,0.0


**Reading the queue.** On this 30,000-item snapshot: `DECLINING_TREND` fires on about 54% of
items (a `trend_direction == "down"` reading, corrected from the Week-6 label bug), `LOW_CTR_GOOD_POSITION`
on about 29%, `HIGH_VOLUME_LOW_VISIBILITY` on about 10%, and `STALE` on under 1%. `LOW_ENGAGEMENT`
did not fire at all on this run — the 25th-percentile engagement rate among pages with any sessions
is itself 0, so the threshold has no room to discriminate; that's a real property of this snapshot,
not a code error, and it's flagged here rather than quietly hidden.

The archetype → action mapping is simple by design: `keyword article` (the dominant content type,
~91% of rows) with a stale/declining reason code routes to a refresh; a page that ranks but doesn't
get clicked routes to a title/meta rewrite; a high-demand keyword with no visibility routes to a
technical audit; `comparison article` always routes to editorial review first, regardless of its
score, because that content type changes underneath you (prices, specs) faster than a quarterly
model refresh can track.

This is **decision-support**: a starting point for a human queue, not a proof that acting on any one
row will move traffic.


## 2. Intended use and limits
Who uses this, for what — and where it stops being valid.

**Intended use.** A weekly-reviewed prioritization list for an editor or SEO strategist deciding
which pages to look at *first*. It orders attention; it does not decide outcomes.

**Limits (in careful words):**
- **Not causal.** The dataset is a single trailing-90-day cross-section, 30,000 items across 32
  clients. Nothing here says refreshing a page *will* improve it — only that these items show
  patterns *associated with* staleness, decline, or low click-through, based on this snapshot.
- **The Week-5 model's probability is unusable.** `refresh_label` was constructed from the same
  columns kept as predictors (`content_age_days`, `ctr`, `impressions_90d`), so its "0.9995
  accuracy" reflects memorizing the labeling rule, not a generalizable signal. It is not used
  anywhere in this playbook.
- **The Week-6 classifier is unvalidated.** Its honest-split F1 of 0 traces to an all-zero target
  (see Section 1) rather than a real generalization test. Until it's re-run with the corrected label
  and re-evaluated on a grouped/time-aware split with a non-trivial precision/recall, it should not
  be reintroduced into ranking.
- **Percent columns are scaled oddly.** `ctr`, `engagement_rate`, `scroll_rate`, and `ai_traffic_pct`
  are already ×100 (e.g. `ctr = 0.76` means 0.76%, not 76%); `scroll_rate` and `ai_traffic_pct` can
  exceed 100 because their numerator/denominator come from different measurement systems — not a
  bug, per the data dictionary.
- **`avg_position == 0` means "no data"**, not top rank — handled explicitly above (`has_data_position`).
- **IDs are pseudonyms.** `content_id`/`client_id` are for grouping only, never used as signal.
- **Single snapshot, single starter dataset.** This does not draw on the ~79M-row warehouse; nothing
  here should be presented as validated against that larger panel.


In [13]:
# Reproducing the claims above, so the markdown isn't just assertion

# 1) Week-5 leakage: the label is a deterministic function of columns kept as features
refresh_label = ((df["content_age_days"] > 180) & (df["ctr"] < 2) & (df["impressions_90d"] > 500)).astype(int)
print("Week-5 refresh_label positive rate:", round(refresh_label.mean(), 3))
print("-> built directly from content_age_days, ctr, impressions_90d, all three also used as features.")
print()

# 2) Week-6 label bug: reproduce the all-zero target
is_declining_buggy = df["trend_direction"].astype(str).str.lower().str.contains("declin").astype(int)
print("Week-6 is_declining (buggy substring match) positive rate:", is_declining_buggy.mean())
print("trend_direction actual categories:", sorted(df['trend_direction'].dropna().unique().tolist()))
print()

# 3) avg_position == 0 ("no data") count
print("Rows with avg_position == 0 (no ranking data):", int((df['avg_position'] == 0).sum()),
      f"({(df['avg_position'] == 0).mean():.1%} of items)")
print()

# 4) percent-scale columns that exceed 100
for c in ["scroll_rate", "ai_traffic_pct"]:
    over = (df[c] > 100).sum()
    print(f"{c}: {over} rows > 100 (measurement-system mismatch, not an error)")


Week-5 refresh_label positive rate: 0.329
-> built directly from content_age_days, ctr, impressions_90d, all three also used as features.

Week-6 is_declining (buggy substring match) positive rate: 0.0
trend_direction actual categories: ['down', 'flat', 'new', 'stable', 'up']

Rows with avg_position == 0 (no ranking data): 1205 (4.0% of items)

scroll_rate: 119 rows > 100 (measurement-system mismatch, not an error)
ai_traffic_pct: 23 rows > 100 (measurement-system mismatch, not an error)


## 3. Human review + the no-go list
What a person must check before acting. What should never be automated.

**Before acting on any row, a human should:**
- Open the actual page and confirm the reason codes still describe reality (the snapshot has an
  age; a page may have already been refreshed since export).
- Check `avg_position == 0` items manually — "no data" is not the same as "needs a technical audit";
  it may mean the page isn't indexed yet, was recently published, or tracking is broken.
- For `comparison article` items, verify prices/specs/claims are current — this content type
  changes independently of the trailing-90-day metrics.
- Confirm the client relationship/contract actually covers content edits before queuing work.
- Check for live campaigns, legal, or compliance-sensitive content before touching anything flagged.

**What must NOT be automated:**
1. **No auto-publish.** No page is rewritten, refreshed, or re-optimized from this queue without a
   human editor reading it first.
2. **No YMYL-style autopilot.** Any page touching pricing, medical, legal, or financial claims
   (`main_intent == "transactional"` or `"commercial"` items included) gets a human read before
   any change, full stop.
3. **No bulk deletion or de-indexing** decided by this queue.
4. **No use of the Week-5 model's raw probability**, anywhere, for any purpose — it's leaked (Section 2).
5. **No client-facing performance guarantees** tied to queue position — this is prioritization, not
   a promise.
6. **No auto-retrain/redeploy.** Any future retrain of the Week-6 classifier must re-run the
   leakage and honest-split checks by hand before its output is trusted again (Section 4).


In [14]:
# Human-review gate: flag items where the reason codes alone aren't enough to act on confidently

def review_reasons(row):
    reasons = []
    if row["avg_position"] == 0:
        reasons.append("no ranking data (avg_position=0) -- verify indexing before treating as low-visibility")
    if row["content_type"] == "comparison article":
        reasons.append("comparison content -- confirm prices/specs are current")
    if pd.isna(row["main_intent"]):
        reasons.append("intent unlabeled -- confirm page purpose before acting")
    if row["main_intent"] in ("transactional", "commercial"):
        reasons.append("commercial/transactional intent -- treat as YMYL-adjacent, human read required")
    return reasons

df["human_review_reasons"]  = df.apply(review_reasons, axis=1)
df["human_review_required"] = df["human_review_reasons"].apply(lambda r: len(r) > 0)

print("Share of all items requiring a human-review flag before action:",
      f"{df['human_review_required'].mean():.1%}")
print()
print("Breakdown of why (an item can carry more than one reason):")
from collections import Counter
flat_reasons = Counter(r.split(" --")[0] for reasons in df["human_review_reasons"] for r in reasons)
for reason, n in flat_reasons.most_common():
    print(f"  {n:>6}  {reason}")


Share of all items requiring a human-review flag before action: 45.6%

Breakdown of why (an item can carry more than one reason):
   10345  commercial/transactional intent
    2374  intent unlabeled
    1205  no ranking data (avg_position=0)
     697  comparison content


## 4. Monitoring / retrain triggers
What would tell you the recommendations went stale?

- **Reason-code fire-rate drift.** If `DECLINING_TREND` share moves sharply from this run's ~54%
  baseline (say, more than 10 points in either direction) between snapshots, that's more likely a
  data-window or tracking change than a real shift in 30,000 pages' fortunes — check the pipeline
  before trusting the new queue.
- **Schema drift.** New/renamed/missing columns (e.g. a new `content_type` category, a renamed tier
  column) will silently break the fixed thresholds above. A column-presence check should run before
  every new ranking (implemented below).
- **Human-review-required share.** If this jumps well above this run's baseline, either the data
  quality degraded (more missing `avg_position`/`main_intent`) or the mix of content types shifted —
  worth a manual look either way.
- **ML-score reintroduction gate (not a schedule).** The Week-6 classifier does not get reintroduced
  into this ranking on a timer. It gets reintroduced only when someone re-runs the corrected
  `trend_direction == "down"` target through a grouped **and** a time-aware split and both show
  precision and recall meaningfully above the ~54% base rate of the positive class — not before.


In [15]:
# --- Schema check: guards against silent breakage from renamed/missing columns ---
EXPECTED_COLS = [
    "content_id", "client_id", "content_type", "main_intent", "search_volume", "cpc",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "days_with_sessions", "trend_direction",
]
missing = [c for c in EXPECTED_COLS if c not in df.columns]
assert not missing, f"Schema drift detected -- missing columns: {missing}"
print("Schema check passed:", len(EXPECTED_COLS), "expected columns all present.")

# --- Save this run's numbers as the baseline the NEXT run should be diffed against ---
monitoring_baseline = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_rows": int(len(df)),
    "reason_code_fire_rate": reason_frame.mean().round(4).to_dict(),
    "declining_trend_share": float((df["trend_direction"] == "down").mean()),
    "human_review_required_share": float(df["human_review_required"].mean()),
    "median_ctr_ranked_pages": float(median_ctr),
    "median_search_volume": float(median_search_volume),
    "ml_score_in_use": False,
    "known_issue_week6_label_bug": (
        "trend_direction values are {down, stable, up, new, flat}; "
        "str.contains('declin') matches 0 of 30000 rows -> Week-6 target was all-zero. "
        "This playbook uses trend_direction == 'down' instead."
    ),
}
print(json.dumps(monitoring_baseline, indent=2))


Schema check passed: 13 expected columns all present.
{
  "generated_at_utc": "2026-08-30T18:25:25.045562+00:00",
  "n_rows": 30000,
  "reason_code_fire_rate": {
    "STALE": 0.0058,
    "DECLINING_TREND": 0.5421,
    "LOW_CTR_GOOD_POSITION": 0.2858,
    "HIGH_VOLUME_LOW_VISIBILITY": 0.1054,
    "LOW_ENGAGEMENT": 0.0
  },
  "declining_trend_share": 0.5420666666666667,
  "human_review_required_share": 0.45613333333333334,
  "median_ctr_ranked_pages": 0.08,
  "median_search_volume": 10.0,
  "ml_score_in_use": false,
  "known_issue_week6_label_bug": "trend_direction values are {down, stable, up, new, flat}; str.contains('declin') matches 0 of 30000 rows -> Week-6 target was all-zero. This playbook uses trend_direction == 'down' instead."
}


## 5. Exports for the paper
Write the queue (and any figures you want to reuse) to `work/outputs/` — your paper builds on
these files.

The ranked queue CSV and this run's monitoring baseline JSON go to `work/outputs/` (the CSV stays
out of git per the leak-guard and regenerates from this notebook; the metrics JSON is small and
meant to be committed as the receipt for next week's numbers). The reason-code figure goes to
`work/figures/` for direct reuse in the paper.


In [16]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1) Ranked queue (CSV) -- regenerated each run, not committed
queue.to_csv("work/outputs/w07_action_queue.csv", index=False)

# 2) Monitoring baseline (JSON) -- small, meant to be committed as the paper's receipt
with open("work/outputs/w07_monitoring_baseline.json", "w") as f:
    json.dump(monitoring_baseline, f, indent=2)

# 3) Figure for reuse in the paper
fig, ax = plt.subplots(figsize=(6, 4))
reason_frame.mean().sort_values().plot(kind="barh", ax=ax, color="#56652F")
ax.set_xlabel("Share of content items")
ax.set_title("Reason-code fire rate across the 30k-item snapshot")
fig.tight_layout()
fig.savefig("work/figures/w07_reason_code_frequency.png", dpi=150)
plt.close(fig)

print("Wrote:")
print(" - work/outputs/w07_action_queue.csv          ", len(queue), "rows")
print(" - work/outputs/w07_monitoring_baseline.json")
print(" - work/figures/w07_reason_code_frequency.png")


Wrote:
 - work/outputs/w07_action_queue.csv           30000 rows
 - work/outputs/w07_monitoring_baseline.json
 - work/figures/w07_reason_code_frequency.png


## Self-check
Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client_id/content_id are pseudonyms, used
      only for grouping)
- [x] My claims use careful words: observed, measured, directional, decision-support — nothing here
      claims the Week-5 or Week-6 model is validated; both are explicitly named as unusable in
      their current form
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
